# 02 - KNN Classifier: EEG Eye State Classification

Trains and evaluates a K-Nearest Neighbors classifier on the multi-band spectral power features built in `01_data_exploration.ipynb`.

Includes hyperparameter tuning (k, weighting, distance metric), full evaluation (accuracy, confusion matrix, ROC, precision-recall),
and diagnostic plots (k-sensitivity curve, PCA projection, learning curve) to understand *why* the model performs the way it does,
not just what its accuracy is.

Run `01_data_exploration.ipynb` first to generate `eeg_features.npz`.


## Setup

In [ ]:
# !pip install mne scikit-learn matplotlib seaborn numpy pandas

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score,
)
from sklearn.decomposition import PCA

SEED = 42
np.random.seed(SEED)


## Load features

Loads the dataset built in `01_data_exploration.ipynb` (run that notebook first if `eeg_features.npz` doesn't exist yet).

In [ ]:
data = np.load("eeg_features.npz", allow_pickle=True)
X, y, ch_names = data["X"], data["y"], data["ch_names"]

print("X shape:", X.shape, "| y shape:", y.shape)
print("Class counts [open, closed]:", np.bincount(y))


## Train/test split and scaling

Z-score normalization matters here more than for tree-based models, since KNN is distance-based.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train_scaled.shape, "| Test shape:", X_test_scaled.shape)
print("Train class counts:", np.bincount(y_train))
print("Test class counts:", np.bincount(y_test))


## Hyperparameter tuning

Grid search over neighborhood size, distance weighting, and distance metric (Manhattan vs. Euclidean).

In [ ]:
param_grid = {
    "n_neighbors": [3, 5, 7, 9, 11],
    "weights": ["uniform", "distance"],
    "p": [1, 2],  # 1 = Manhattan, 2 = Euclidean
}

grid = GridSearchCV(
    KNeighborsClassifier(metric="minkowski"),
    param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,
)
grid.fit(X_train_scaled, y_train)

print("Best params:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)

best_knn = grid.best_estimator_


## Test-set evaluation

In [ ]:
y_pred = best_knn.predict(X_test_scaled)

test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {test_accuracy:.4f}")
print("\nClassification report:\n", classification_report(y_test, y_pred))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["True: Open (0)", "True: Closed (1)"],
    columns=["Pred: Open (0)", "Pred: Closed (1)"],
)
print(cm_df)

plt.figure(figsize=(5, 4))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - Optimized KNN (Eyes Open vs Closed)")
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.savefig("knn_confusion_matrix.png", dpi=150)
plt.show()


## Diagnostics

Beyond raw accuracy: how sensitive is performance to k, how separable are the classes, and is the model over- or under-fitting?

In [ ]:
# Accuracy vs. k
accuracies = []
k_values = range(1, 25)
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    accuracies.append(knn.score(X_test_scaled, y_test))

plt.figure(figsize=(6, 4))
plt.plot(k_values, accuracies, marker="o")
plt.xlabel("k value")
plt.ylabel("Accuracy")
plt.title("KNN Sensitivity to Number of Neighbors")
plt.grid(True)
plt.tight_layout()
plt.savefig("knn_k_sensitivity.png", dpi=150)
plt.show()


In [ ]:
y_proba = best_knn.predict_proba(X_test_scaled)[:, 1]

fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (KNN)")
plt.legend()
plt.tight_layout()
plt.savefig("knn_roc_curve.png", dpi=150)
plt.show()


In [ ]:
precision, recall, _ = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)

plt.figure()
plt.plot(recall, precision, label=f"AP = {ap:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve (KNN)")
plt.legend()
plt.tight_layout()
plt.savefig("knn_precision_recall.png", dpi=150)
plt.show()


In [ ]:
nn = NearestNeighbors(n_neighbors=best_knn.n_neighbors)
nn.fit(X_train_scaled)
distances, _ = nn.kneighbors(X_test_scaled)
mean_dist = distances.mean(axis=1)
correct = y_pred == y_test

plt.figure()
plt.hist(mean_dist[correct], bins=30, alpha=0.7, label="Correct")
plt.hist(mean_dist[~correct], bins=30, alpha=0.7, label="Incorrect")
plt.xlabel("Mean Distance to Neighbors")
plt.ylabel("Count")
plt.title("Neighbor Distance: Correct vs Incorrect Predictions")
plt.legend()
plt.tight_layout()
plt.savefig("knn_neighbor_distance.png", dpi=150)
plt.show()


In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_test_scaled)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y_test, alpha=0.6)
axes[0].set_title("PCA Projection (True Labels)")
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")

axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y_pred, alpha=0.6)
axes[1].set_title("PCA Projection (Predicted Labels)")
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")

plt.tight_layout()
plt.savefig("knn_pca_projection.png", dpi=150)
plt.show()


In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    best_knn, X_train_scaled, y_train, cv=5, scoring="accuracy",
    train_sizes=np.linspace(0.1, 1.0, 6),
)

plt.figure()
plt.plot(train_sizes, train_scores.mean(axis=1), label="Train")
plt.plot(train_sizes, val_scores.mean(axis=1), label="Validation")
plt.xlabel("Training Samples")
plt.ylabel("Accuracy")
plt.title("Learning Curve (KNN)")
plt.legend()
plt.tight_layout()
plt.savefig("knn_learning_curve.png", dpi=150)
plt.show()
